In [ ]:
# 2. Define File Paths and Home Directory
USER_HOME = os.path.expanduser('~')
WORKSPACE_ROOT = os.getcwd() # Assumption: Notebook running in workspace or close to it
print(f"User Home: {USER_HOME}")

# Targets
SETTINGS_PATH = os.path.join(USER_HOME, "Library/Application Support/Code/User/settings.json")
# For the purpose of this notebook, we generate 'clean' versions in the local directory first
OUTPUT_DIR = os.path.join(WORKSPACE_ROOT, "generated_configs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Target Settings Path: {SETTINGS_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")

In [ ]:
# 3. Audit VS Code Settings (Mockup reading, real sanitization logic)
# We will define the cleaning logic here.

def sanitize_json_paths(data, home_dir):
    json_str = json.dumps(data, indent=4)
    # Replace /Users/username with ${userHome} for VS Code settings context usually,
    # but specific terminal args usually take $HOME or just ~

    # Simple regex to replace exact home path
    sanitized = json_str.replace(home_dir, "${userHome}")

    return json.loads(sanitized)

# Because we can't fully read the external settings.json securely in all envs without permissions,
# We will create the *Standardized* Profile block structure here to be copied.

standard_profiles = {
    "terminal.integrated.profiles.osx": {
        "bash": {
            "path": "bash",
            "args": ["-l", "-c", "cd $HOME && exec bash"],
            "icon": "terminal",
            "color": "terminal.ansiWhite"
        },
        "zsh": {
            "path": "zsh",
            "args": ["-l"],
            "icon": "terminal",
            "color": "terminal.ansiWhite"
        },
        "GenAI-RD": {
            "path": "zsh",
            "args": ["-c", "cd ${workspaceFolder} && pwd && exec zsh -l"],
            "icon": "hubot",
            "color": "terminal.ansiCyan"
        },
        "LLM-Council": {
             "path": "zsh",
             "args": ["-c", "cd ${userHome}/www/LLM-Council && pwd && exec zsh -l"],
             "icon": "debug-console",
             "color": "terminal.ansiYellow"
        }
        # Add others with ${userHome}
    }
}

print("Standardized Profiles defined.")

In [ ]:
# 6. Sanitize Shell Configs (bash, zsh, fish, pwsh)
# Generating clean, portable snippets.

bash_rc_content = """
# Bash Configuration
# Portable aliases
alias ll='ls -FGlAhp'
alias work='cd $HOME/workspace'

# Repository Shortcuts
alias go-genai='cd $HOME/workspace/GenAI-RD'
""".strip()

zsh_rc_content = """
# Zsh Configuration
# Portable aliases
alias ll='ls -FGlAhp'
alias work='cd $HOME/workspace'

# Repository Shortcuts
alias go-genai='cd $HOME/workspace/GenAI-RD'

# Prompt (Simple Git Aware)
autoload -Uz vcs_info
precmd() { vcs_info }
zstyle ':vcs_info:git:*' formats '%b'
setopt PROMPT_SUBST
PROMPT='%F{cyan}%~%f %F{yellow}(${vcs_info_msg_0_})%f %# '
""".strip()

fish_config_content = """
# Fish Configuration
alias ll 'ls -FGlAhp'
alias work 'cd $HOME/workspace'

# Repository Shortcuts
alias go-genai 'cd $HOME/workspace/GenAI-RD'
""".strip()

pwsh_profile_content = """
# PowerShell Profile
function go-genai { Set-Location "$HOME/workspace/GenAI-RD" }
function work { Set-Location "$HOME/workspace" }
""".strip()

print("Shell configurations generated in memory.")

In [ ]:
# 8. Write Sanitized Configurations to Disk (in code/term/)
target_dir = os.path.join(WORKSPACE_ROOT, "..", "term") # Adjusting based on current file location assumption
# Let's verify where we are
print(f"Current Dir: {os.getcwd()}")
# Force correct location based on notebook path /Users/bamdad/www/GenAI-RD/code/term/
# We are IN code/term/

with open("bashrc_snippet", "w") as f:
    f.write(bash_rc_content)

with open("zshrc_snippet", "w") as f:
    f.write(zsh_rc_content)

with open("config.fish_snippet", "w") as f:
    f.write(fish_config_content)

with open("Microsoft.PowerShell_profile.ps1_snippet", "w") as f:
    f.write(pwsh_profile_content)

print("Files written to code/term/")